In [17]:
from astropy.io import fits
from pylab import *
from scipy.interpolate import RegularGridInterpolator
import os
import pandas as pd

In [18]:
def makefilename(offset,Rh,i):
    return 'img_s'+str(offset).zfill(4)+'_Rh'+str(Rh)+'_i'+str(i)+".fits"

In [19]:
Rh_list = [1, 10, 40, 160]
i_list = [10, 30, 50, 70, 90, 110, 130, 150, 170]
offsets = range(1500, 3000)  # Pas de stapgrootte (hier 10) aan naar jouw data!
r_grid = linspace(0, 200, 200)
phi_grid = linspace(0, 2*pi, 360)
R, PHI = meshgrid(r_grid, phi_grid)
X_prime = R * cos(PHI)
Y_prime = R * sin(PHI)
pts = column_stack((Y_prime.ravel(), X_prime.ravel()))

In [20]:
dirs = {
    'retro': '/zfs/helios/filer0/oporth/share/MAD-rhovfix/a-15o16',
    'pro': '/zfs/helios/filer0/oporth/share/MAD-rhovfix/a+15o16'
}

results = []

In [21]:
# --- LUS DOOR HET GRID ---
for Rh in Rh_list:
    for i in i_list:
        print(f"Processing Rh={Rh}, i={i}...")
        
        # Sla de profielen op om later metrics te berekenen
        profiles = {'retro': None, 'pro': None}
        
        for spin_type, base_dir in dirs.items():
            all_profiles = []
            
            for offset in offsets:
                filepath = os.path.join(base_dir, 'fits', makefilename(offset, Rh, i))
                if not os.path.exists(filepath):
                    continue
                    
                with fits.open(filepath) as hdul:
                    img_data = hdul[0].data[0, :, :]
                    
                    # Zorg ervoor dat de byte-order klopt voor SciPy/Cython
                    if img_data.dtype.byteorder in ('>', 'B') or (img_data.dtype.byteorder == '=' and sys.byteorder == 'big'):
                        img = img_data.byteswap().newbyteorder()
                    else:
                        img = img_data
                
                x_orig = arange(img.shape[1]) - 200
                y_orig = arange(img.shape[0]) - 200
                interp = RegularGridInterpolator((y_orig, x_orig), img, bounds_error=False, fill_value=0)
                I_r_phi = interp(pts).reshape(R.shape)
                
                mean_I_r = mean(I_r_phi, axis=0)
                mean_I_r_norm = mean_I_r / sum(img)  # Normalisatie per snapshot
                all_profiles.append(mean_I_r_norm)
            
            if len(all_profiles) > 0:
                profiles[spin_type] = mean(all_profiles, axis=0)
        
        # Als beide mappen data hadden, bereken de metrics
        if profiles['retro'] is not None and profiles['pro'] is not None:
            # Centroids
            c_retro = sum(r_grid * profiles['retro']) / sum(profiles['retro'])
            c_pro = sum(r_grid * profiles['pro']) / sum(profiles['pro'])
            
            # Piek radii
            p_retro = r_grid[argmax(profiles['retro'])]
            p_pro = r_grid[argmax(profiles['pro'])]
            
            # DE CRUCIALE METRIC: Hoeveel is het zwaartepunt inwaarts verschoven?
            # Als dit dicht bij 0 ligt, is het profiel gestagneerd (Jouw hypothese!)
            centroid_shift = c_retro - c_pro
            
            results.append({
                'Rh': Rh,
                'inclination': i,
                'centroid_retro': c_retro,
                'centroid_pro': c_pro,
                'centroid_shift': centroid_shift,
                'peak_retro': p_retro,
                'peak_pro': p_pro
            })

Processing Rh=1, i=10...


KeyboardInterrupt: 

In [ ]:
# Opslaan als CSV voor makkelijke analyse/plotting
df = pd.DataFrame(results)
df.to_csv('output_intensity/grid_hypothesis_results.csv', index=False)
print("Grid analyse voltooid! Resultaten opgeslagen in grid_hypothesis_results.csv")

Grid analyse voltooid! Resultaten opgeslagen in grid_hypothesis_results.csv
